In [2]:
print("ok")

ok


In [3]:

from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter

import os
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain

In [4]:
%pwd


'c:\\Users\\vaish\\Source-code-Analyzer\\research'

In [5]:
!mkdir test_repo

A subdirectory or file test_repo already exists.


In [6]:
repo_path = "test_repo/"

repo = Repo.clone_from("https://github.com/VJ70/MLFlow", to_path=repo_path)

GitCommandError: Cmd('git') failed due to: exit code(128)
  cmdline: git clone -v -- https://github.com/VJ70/MLFlow test_repo/
  stderr: 'fatal: destination path 'test_repo' already exists and is not an empty directory.
'

In [7]:
loader = GenericLoader.from_filesystem(repo_path,
                                        glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

In [8]:
documents = loader.load()
documents

[Document(page_content='from flask import Flask, render_template, request\nimport os\nimport numpy as np\nimport pandas as pd\nfrom src.MlopsProject.pipeline.prediction import PredictionPipeline\n\napp = Flask(__name__) # initializing a flask app\n\n\n@app.route(\'/\',methods=[\'GET\'])  # route to display the home page\ndef homePage():\n    return render_template("index.html")\n\n\n@app.route(\'/train\',methods=[\'GET\'])  # route to train the pipeline\ndef training():\n    os.system("python main.py")\n    return "Training Successful!" \n\n\n@app.route(\'/predict\',methods=[\'POST\',\'GET\']) # route to show the predictions in a web UI\ndef index():\n    if request.method == \'POST\':\n        try:\n            #  reading the inputs given by the user\n            fixed_acidity =float(request.form[\'fixed_acidity\'])\n            volatile_acidity =float(request.form[\'volatile_acidity\'])\n            citric_acid =float(request.form[\'citric_acid\'])\n            residual_sugar =float(

In [9]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language = Language.PYTHON,
                                                             chunk_size = 500,
                                                             chunk_overlap = 20)

In [10]:
texts = documents_splitter.split_documents(documents)
texts

[Document(page_content='from flask import Flask, render_template, request\nimport os\nimport numpy as np\nimport pandas as pd\nfrom src.MlopsProject.pipeline.prediction import PredictionPipeline\n\napp = Flask(__name__) # initializing a flask app\n\n\n@app.route(\'/\',methods=[\'GET\'])  # route to display the home page\ndef homePage():\n    return render_template("index.html")\n\n\n@app.route(\'/train\',methods=[\'GET\'])  # route to train the pipeline', metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}),
 Document(page_content='def training():\n    os.system("python main.py")\n    return "Training Successful!" \n\n\n@app.route(\'/predict\',methods=[\'POST\',\'GET\']) # route to show the predictions in a web UI', metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}),
 Document(page_content="def index():\n    if request.method == 'POST':\n        try:\n            #  reading the inputs given by the user\n            fixed_aci

In [11]:

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [12]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [13]:

embeddings=OpenAIEmbeddings(disallowed_special=())

In [ ]:
vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory='./db')

In [ ]:

vectordb.persist()

In [ ]:
llm = ChatOpenAI()

In [ ]:
memory = ConversationSummaryMemory(llm=llm, memory_key = "chat_history", return_messages=True)

In [ ]:
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k":3}), memory=memory)

In [ ]:
question = "what is function doing?"